In [ ]:
import gymnasium as gym
from my_USV_Env import MyUsvEnv_Gym
# Register the environment so we can create it with gym.make()
gym.register(
    id="myenv/MyUsvEnv_Gym-v0",
    entry_point=MyUsvEnv_Gym,
)
# 2. 创建环境实例
env = gym.make("myenv/MyUsvEnv_Gym-v0")  # 使用注册的id创建环境
from gymnasium.utils.env_checker import check_env
# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

In [ ]:
from stable_baselines3 import TD3
from stable_baselines3.common.env_util import make_vec_env
# Parallel environments
vec_env = make_vec_env("myenv/MyUsvEnv_Gym-v0", n_envs=4)
model = TD3("MultiInputPolicy", vec_env, verbose=1)
model.learn(total_timesteps=250000)
model.save("td3_MyUsvEnv")

In [ ]:
# 创建测试环境（使用与训练相同的环境参数）
from stable_baselines3 import TD3
import gymnasium as gym
import matplotlib.pyplot as plt
from my_USV_Env import MAX, MyUsvEnv_Gym
model = TD3.load("td3_MyUsvEnv")

# Register the environment so we can create it with gym.make()
gym.register(
    id="myenv/MyUsvEnv_Gym-v0",
    entry_point=MyUsvEnv_Gym,
)
env = gym.make("myenv/MyUsvEnv_Gym-v0")
env.unwrapped.__init__(record_flag=True) 
obs, info = env.reset()  # 新版Gym返回元组
terminated = False
truncated = False
total_reward = 0

while not (terminated or truncated):
    action, _ = model.predict(obs, deterministic=True)  # 确保传入的是numpy数组
    # 新版Gym返回5个值
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    points=(float(obs["x_norm"]*MAX),float(obs["y_norm"]*MAX))
    # env.record_points(points)
    env.unwrapped.record_points(points,csv_file="usv_path_points_2.csv")
print(total_reward)    
env.unwrapped.draw_path("usv_path_points_2.csv")


